# The Syndicate - Real-Time Credit Card Fraud Detector

End-to-end pipeline: load the imbalanced transaction data, explore the class imbalance,
preprocess features, balance the training set with **SMOTE**, train a **Random Forest**
classifier, evaluate it with imbalance-aware metrics, and save the trained model artifact
used by `scripts/realtime_inference.py`.

**Dataset:** [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)
(ULB Machine Learning Group, via Kaggle) - 284,807 European cardholder transactions from
September 2013, 492 of which are fraudulent (~0.172%). Features `V1`-`V28` are PCA components
of the original (confidential) transaction features; `Time` and `Amount` are the only
untransformed columns; `Class` is the target (1 = fraud).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    roc_auc_score,
    roc_curve,
    average_precision_score,
    precision_recall_curve,
)
from imblearn.over_sampling import SMOTE
import joblib

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

print("Libraries imported successfully.")

## 1. Load the data

In [ ]:
df = pd.read_csv("../data/creditcard.csv")

print("Dataset shape:", df.shape)
print("\nFirst 5 rows:")
display(df.head())

print("\nMissing values per column:")
print(df.isnull().sum().sum(), "total missing values")

## 2. Explore the class imbalance

This is the core challenge of the dataset: fraud is extremely rare relative to genuine transactions.

In [ ]:
class_counts = df["Class"].value_counts()
class_pct = df["Class"].value_counts(normalize=True) * 100

print("Class distribution:")
print(class_counts)
print("\nClass distribution (%):")
print(class_pct.round(4))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.countplot(x="Class", data=df, ax=axes[0])
axes[0].set_title("Transaction counts by class")
axes[0].set_xticklabels(["Genuine (0)", "Fraud (1)"])

axes[1].pie(
    class_counts,
    labels=["Genuine", "Fraud"],
    autopct="%1.3f%%",
    startangle=90,
    colors=["#4C72B0", "#C44E52"],
)
axes[1].set_title("Class share")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["Amount"], bins=50, ax=axes[0])
axes[0].set_title("Transaction amount distribution")
axes[0].set_yscale("log")

sns.histplot(df["Time"], bins=50, ax=axes[1])
axes[1].set_title("Transaction time distribution (seconds since first transaction)")

plt.tight_layout()
plt.show()

print(df[["Time", "Amount"]].describe())

## 3. Preprocess: split, then scale

`Time` and `Amount` are the only features not already on a comparable PCA scale, so those are the only two columns standardized. The scaler is fit **only on the training split** to avoid leaking test-set statistics.

In [ ]:
X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print("Train shape:", X_train.shape, "| Fraud rate:", y_train.mean().round(5))
print("Test shape: ", X_test.shape, "| Fraud rate:", y_test.mean().round(5))

scaler = StandardScaler()
X_train[["Time", "Amount"]] = scaler.fit_transform(X_train[["Time", "Amount"]])
X_test[["Time", "Amount"]] = scaler.transform(X_test[["Time", "Amount"]])

X_train[["Time", "Amount"]].describe()

## 4. Balance the training set with SMOTE

SMOTE (Synthetic Minority Over-sampling Technique) generates synthetic fraud examples by interpolating between existing fraud cases and their nearest fraud neighbors, rather than just duplicating them. **Critically, SMOTE is applied only to the training set, after the train/test split** - the test set must stay untouched and reflect the real-world imbalance, or evaluation metrics become meaningless.

In [ ]:
print("Before SMOTE:")
print(y_train.value_counts())

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print("\nAfter SMOTE:")
print(y_train_res.value_counts())
print("\nResampled training shape:", X_train_res.shape)

## 5. Train a Random Forest classifier

`class_weight="balanced"` is kept as a second, complementary safeguard on top of SMOTE so the trees still penalize missed fraud more heavily during splits.

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced",
)

model.fit(X_train_res, y_train_res)
print("Model trained.")

## 6. Evaluate

Accuracy is meaningless on a 0.17%-fraud dataset (predicting "genuine" every time gives ~99.8% accuracy). Instead we look at precision/recall on the fraud class, the confusion matrix, ROC-AUC, and PR-AUC (more informative than ROC-AUC under heavy imbalance).

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Classification report:")
print(classification_report(y_test, y_pred, target_names=["Genuine", "Fraud"]))

roc_auc = roc_auc_score(y_test, y_proba)
pr_auc = average_precision_score(y_test, y_proba)
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC:  {pr_auc:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["Genuine", "Fraud"]).plot(ax=axes[0], colorbar=False)
axes[0].set_title("Confusion matrix")

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, label=f"ROC-AUC = {roc_auc:.4f}")
axes[1].plot([0, 1], [0, 1], "k--", alpha=0.4)
axes[1].set_xlabel("False positive rate")
axes[1].set_ylabel("True positive rate")
axes[1].set_title("ROC curve")
axes[1].legend()

prec, rec, _ = precision_recall_curve(y_test, y_proba)
axes[2].plot(rec, prec, label=f"PR-AUC = {pr_auc:.4f}")
axes[2].set_xlabel("Recall")
axes[2].set_ylabel("Precision")
axes[2].set_title("Precision-Recall curve")
axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(model.feature_importances_, index=X_train.columns)
importances.sort_values(ascending=False).head(15).plot(kind="barh", figsize=(8, 6))
plt.gca().invert_yaxis()
plt.title("Top 15 feature importances")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 7. Save the model artifact

Bundles the fitted model together with the fitted `StandardScaler`, since `scripts/realtime_inference.py` needs both to score a raw incoming transaction the same way the training data was scaled.

In [ ]:
import os
os.makedirs("../models", exist_ok=True)

joblib.dump(
    {"model": model, "scaler": scaler},
    "../models/fraud_model.joblib",
)

print("Saved to ../models/fraud_model.joblib")

## 8. Real-time inference

The saved artifact is consumed by `scripts/realtime_inference.py`, which simulates a stream
of incoming transactions, scores each one's fraud probability, and prints a console alert
whenever that probability exceeds **95%**. Run it from the project root:

```bash
python scripts/realtime_inference.py
```
